# 02. Analisis del Piloto

## Objetivo

Evaluar el desempeño del piloto de tarjetas de débito físicas y comparar el comportamiento de los clientes que recibieron una tarjeta física frente a quienes no la recibieron.

El análisis busca:

- identificar la adopción y activación de las tarjetas físicas;
- medir cambios en frecuencia y monto transaccional;
- comparar el comportamiento entre clientes con y sin tarjeta física;
- definir las bases analíticas que posteriormente permitirán construir un criterio de priorización para la siguiente ola de entrega.

## 0. Configuracion

In [1]:
# Imports
# Configura las librerías necesarias para leer los datasets Silver desde S3.

from io import BytesIO

import boto3
import pandas as pd

In [2]:
# Configuración AWS
# Define el perfil local y la ubicación de la capa Silver.

AWS_PROFILE = "ds-technical-test"
S3_BUCKET = "bg-ds-debit-card-pilot-bucket"
SILVER_PREFIX = "silver"

session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client("s3")

In [3]:
# Lectura de datasets Silver
# Lee todos los archivos Parquet de un prefijo de S3 y los consolida en un DataFrame.

def read_s3_parquet_folder(folder: str) -> pd.DataFrame:
    prefix = f"{SILVER_PREFIX}/{folder}/"

    paginator = s3.get_paginator("list_objects_v2")
    parquet_files = []

    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        parquet_files.extend(
            obj["Key"]
            for obj in page.get("Contents", [])
            if obj["Key"].endswith(".parquet")
        )

    dataframes = []

    for key in parquet_files:
        response = s3.get_object(Bucket=S3_BUCKET, Key=key)
        dataframes.append(
            pd.read_parquet(BytesIO(response["Body"].read()))
        )

    return pd.concat(dataframes, ignore_index=True)

In [4]:
# Carga de Silver
# Recupera las cinco fuentes procesadas y estandarizadas por el Glue ETL Job.

customers = read_s3_parquet_folder("customers")
cards = read_s3_parquet_folder("cards")
transactions = read_s3_parquet_folder("transactions")
marketing_interactions = read_s3_parquet_folder("marketing_interactions")
merchant_catalog = read_s3_parquet_folder("merchant_catalog")

In [ ]:
# Validación de carga

datasets = {
    "customers": customers,
    "cards": cards,
    "transactions": transactions,
    "marketing_interactions": marketing_interactions,
    "merchant_catalog": merchant_catalog,
}

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows")

customers: 12,000 rows
cards: 16,585 rows
transactions: 193,015 rows
marketing_interactions: 36,000 rows
merchant_catalog: 42 rows


## 1. Definicion del Piloto

### 1.1. Identificacion de tarjetas fisicas y virtuales

In [6]:
# Distribución de tipos de tarjeta

display(
    cards["tipo"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
tipo,
virtual,10719
fisica,5011
v,484
f,191
física,180


## 2. Desempenio del Piloto

## 3. Conclusiones del Piloto